<a href="https://colab.research.google.com/github/Floriannee/LLM-TTT/blob/main/TTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch accelerate bitsandbytes

In [4]:
import torch
import re
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. CHARGEMENT DU MODÈLE QWEN

print("Chargement du modèle Qwen... (Cela peut prendre quelques minutes)")

model_id = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Modèle chargé avec succès !\n")


# 2. LOGIQUE DU JEU (TIC TAC TOE)

def print_board(board):
    print(f"\n {board[0]} | {board[1]} | {board[2]} ")
    print("---+---+---")
    print(f" {board[3]} | {board[4]} | {board[5]} ")
    print("---+---+---")
    print(f" {board[6]} | {board[7]} | {board[8]} \n")

def check_win(board, player):
    win_conditions = [
        [0,1,2], [3,4,5], [6,7,8], # Lignes
        [0,3,6], [1,4,7], [2,5,8], # Colonnes
        [0,4,8], [2,4,6]           # Diagonales
    ]
    return any(all(board[i] == player for i in combo) for combo in win_conditions)

def check_draw(board):
    return ' ' not in board

def get_available_moves(board):
    return [str(i+1) for i in range(9) if board[i] == ' ']


# 3. INTERACTION AVEC QWEN

def get_qwen_move(board, qwen_mark):
    available = get_available_moves(board)

    # Création de la représentation textuelle du plateau pour le prompt
    b = board
    board_visual = f"""
     {b[0]} | {b[1]} | {b[2]}
    -----------
     {b[3]} | {b[4]} | {b[5]}
    -----------
     {b[6]} | {b[7]} | {b[8]}
    """

    # Utilisation du "Chat Template" de Qwen pour de meilleurs résultats
    messages = [
        {"role": "system", "content": "Tu es une IA qui joue au Tic-Tac-Toe. Tu dois absolument répondre uniquement avec le numéro de ta case (de 1 à 9) au format exact : MOVE: <numéro>. Ne donne aucune explication."},
        {"role": "user", "content": f"Plateau actuel :\n{board_visual}\nTu joues les '{qwen_mark}'. Cases disponibles : {', '.join(available)}. À ton tour. Réponds avec MOVE: <numéro>"}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Génération
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20, do_sample=True, temperature=0.7)

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Parsing de la réponse pour extraire le numéro
    match = re.search(r'MOVE:\s*(\d)', response)
    if match:
        move = int(match.group(1))
        if 1 <= move <= 9 and str(move) in available:
            return move - 1 # Retourne l'index (0-8)

    # Fallback : si le LLM n'a pas compris, on choisit au hasard pour ne pas bloquer
    print(f"(Note: Qwen a répondu '{response.strip()}', ce qui est invalide. Coup aléatoire choisi.)")
    return int(random.choice(available)) - 1

# 4. BOUCLE DE JEU PRINCIPALE

def play_game():
    board = [' '] * 9
    print("Bienvenue ! Tu joues les 'X', Qwen joue les 'O'.")
    print("Les cases sont numérotées de 1 à 9 (de gauche à droite, de haut en bas).")
    print_board([str(i+1) for i in range(9)])

    current_turn = 'X' # Le joueur humain commence

    while True:
        if current_turn == 'X':
            while True:
                try:
                    move = int(input("Ton tour (1-9) : ")) - 1
                    if 0 <= move <= 8 and board[move] == ' ':
                        break
                    else:
                        print("Case invalide ou déjà prise. Réessaie.")
                except ValueError:
                    print("Entre un nombre entre 1 et 9.")
            board[move] = 'X'
        else:
            print("Qwen réfléchit...")
            move = get_qwen_move(board, 'O')
            board[move] = 'O'
            print(f"Qwen joue la case {move + 1}.")

        print_board(board)

        if check_win(board, current_turn):
            print(f"{current_turn} a gagné !")
            break
        if check_draw(board):
            print("Match nul !")
            break

        # Changer de tour
        current_turn = 'O' if current_turn == 'X' else 'X'



Chargement du modèle Qwen... (Cela peut prendre quelques minutes)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Modèle chargé avec succès !



In [5]:
# Lancer le jeu
play_game()

Bienvenue ! Tu joues les 'X', Qwen joue les 'O'.
Les cases sont numérotées de 1 à 9 (de gauche à droite, de haut en bas).

 1 | 2 | 3 
---+---+---
 4 | 5 | 6 
---+---+---
 7 | 8 | 9 

Ton tour (1-9) : 1

 X |   |   
---+---+---
   |   |   
---+---+---
   |   |   

Qwen réfléchit...
Qwen joue la case 5.

 X |   |   
---+---+---
   | O |   
---+---+---
   |   |   

Ton tour (1-9) : 3

 X |   | X 
---+---+---
   | O |   
---+---+---
   |   |   

Qwen réfléchit...
Qwen joue la case 4.

 X |   | X 
---+---+---
 O | O |   
---+---+---
   |   |   

Ton tour (1-9) : 2

 X | X | X 
---+---+---
 O | O |   
---+---+---
   |   |   

X a gagné !
